# LandslideGuard - Stage 2: Detection Model Development

Trains and evaluates a 14-channel U-Net on Landslide4Sense. Runs on Kaggle GPU.

**Stage 1 is frozen** - this notebook imports `src.detection.preprocessing`,
`src.detection.dataset`, and the train-only normalization statistics from
`outputs/detection/data_verification/normalization_statistics.json`. Nothing
here recomputes them.

**Test set is locked** - it is evaluated exactly once, in Section 22, after
the model + threshold have been chosen from validation only.

Set `GITHUB_REPO` and `DATA_ROOT` in Section 01, then Runtime -> Run All.


## SECTION 01 - Stage-2 Configuration

All tunables live in `configs/detection.yaml`. Two fields must be edited on
Kaggle: `GITHUB_REPO` and `DATA_ROOT`. Small overrides for a quick sanity
run (e.g. `OVERRIDE_EPOCHS = 3`) are exposed below the config cell.


In [ ]:
# ==== EDIT THESE TWO ====
GITHUB_REPO = "https://github.com/Aryan2080/Landslide-Guard.git"
GITHUB_REF  = "main"
DATA_ROOT   = "/kaggle/input/landslide4sense"
# =========================

# Optional overrides (leave None to use configs/detection.yaml)
OVERRIDE_EPOCHS         = None    # e.g. 5 for a smoke run
OVERRIDE_BATCH_SIZE     = None
OVERRIDE_LR             = None
OVERRIDE_BASE_FEATURES  = None

# Class-imbalance / HP experiments (Sections 16-17) run shorter than the
# baseline. Set to a small number for a fast pass, larger for real work.
EXPERIMENT_EPOCHS       = 10

SEED = 42


## SECTION 02 - Stage-1 Pipeline Verification

Clones the LandslideGuard repo, adds it to `sys.path`, and imports the
frozen Stage-1 modules. Also loads the frozen train-only normalization
statistics (never recomputed). Aborts if anything is wrong.


In [ ]:
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO_DIR = WORK / "LandslideGuard"

def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout: print(r.stdout.rstrip())
    if r.stderr: print(r.stderr.rstrip())
    if r.returncode != 0:
        raise RuntimeError(f"command failed: exit {r.returncode}")

if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    run(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR)
    run(["git", "checkout", GITHUB_REF],       cwd=REPO_DIR)
    run(["git", "pull", "--ff-only"],          cwd=REPO_DIR)
else:
    if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--depth=1", "--branch", GITHUB_REF, GITHUB_REPO, str(REPO_DIR)])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.run(["git","rev-parse","HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
print("REPO HEAD:", commit)

# --- Imports from the frozen Stage-1 pipeline ---
from src.detection.preprocessing import (
    N_CHANNELS, NormalizationStats, preprocess_pair,
    read_image, read_mask, sanitize, normalize)
from src.detection.dataset       import (
    Landslide4SenseDataset, build_dataloader, build_all_splits)

# --- Imports from Stage-2 code ---
from src.detection.model         import UNet, count_parameters
from src.detection.losses        import (
    DiceLoss, BCEDiceLoss, FocalLoss, FocalDiceLoss,
    BCEWithLogitsLossAligned, build_loss)
from src.detection.metrics       import (
    BinaryMetricAccumulator, PRAUCAccumulator, sweep_thresholds)
from src.detection.train         import Trainer, fit, evaluate, RunHistory
from src.detection.validate      import compute_metrics, sweep_threshold
from src.detection.utils         import (
    set_seed, seeded_worker_init, device_summary, load_config, save_config,
    ExperimentTracker, ExperimentRow, EarlyStopping)
from src.detection.postprocessing import (
    probability_to_mask, remove_small_components, fill_small_holes,
    extract_boundaries, PostprocessingConfig, apply as apply_postproc)
from src.detection.inference     import DetectionInference

set_seed(SEED)

# --- Load config ---
cfg = load_config(REPO_DIR / "configs" / "detection.yaml")
model_cfg = cfg["model"]
train_cfg = cfg["training"]
loss_cfg  = train_cfg["loss"]
infer_cfg = cfg["inference"]

if OVERRIDE_EPOCHS        is not None: train_cfg["epochs"]         = int(OVERRIDE_EPOCHS)
if OVERRIDE_BATCH_SIZE    is not None: train_cfg["batch_size"]     = int(OVERRIDE_BATCH_SIZE)
if OVERRIDE_LR            is not None: train_cfg["learning_rate"]  = float(OVERRIDE_LR)
if OVERRIDE_BASE_FEATURES is not None: model_cfg["base_features"]  = int(OVERRIDE_BASE_FEATURES)

# --- Load frozen normalization statistics (train-only) ---
norm_path = REPO_DIR / "outputs" / "detection" / "data_verification" / "normalization_statistics.json"
assert norm_path.is_file(), f"missing frozen normalization stats: {norm_path}"
norm_payload = json.loads(norm_path.read_text())
assert norm_payload["computed_on"] == "train_split_only", \
       "STAGE-1 CONTRACT VIOLATION - normalization stats not train-only"
assert len(norm_payload["mean"]) == N_CHANNELS == 14
stats = NormalizationStats.from_json(norm_path)
print("normalization: method=", norm_payload["method"],
      "  computed_on=", norm_payload["computed_on"],
      "  n_train_files=", norm_payload["n_train_files"])

# --- Torch / device ---
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device_summary(DEVICE))
if DEVICE.type != "cuda":
    print("WARNING: CUDA not available. Stage 2 is designed for Kaggle GPU.")

print("STAGE-1 INTERFACE CHECK: PASS")


## SECTION 03 - Dataset / DataLoader Sanity Check

Instantiates the three official splits, verifies file counts, and checks
that one real batch matches the Stage-1 contract:

* image `(B, 14, 128, 128) float32` finite
* mask `(B, 128, 128) float32` values in `{0.0, 1.0}`
* validation and test have no augmentation


In [ ]:
root = Path(DATA_ROOT)
assert root.is_dir(), f"DATA_ROOT does not exist: {root}"

def resolve(root, split):
    for img_d, mask_d in [(root/split/"img", root/split/"mask"),
                          (root/split/split/"img", root/split/split/"mask")]:
        if img_d.is_dir() and mask_d.is_dir():
            return img_d, mask_d
    raise FileNotFoundError(f"could not find img/ + mask/ under {root/split}")

paths = {name: resolve(root, name) for name in ["TrainData","ValidData","TestData"]}

ds_train = Landslide4SenseDataset(*paths["TrainData"], stats=stats,
                                  split="train", augment_seed=SEED)
ds_valid = Landslide4SenseDataset(*paths["ValidData"], stats=stats, split="valid")
ds_test  = Landslide4SenseDataset(*paths["TestData"],  stats=stats, split="test")

EXPECTED = {"train": 3799, "valid": 245, "test": 800}
for name, d in [("train", ds_train), ("valid", ds_valid), ("test", ds_test)]:
    r = d.report
    assert r.n_paired == EXPECTED[name], (name, r.n_paired)
    print(f"{name:5s} paired={r.n_paired:4d}  augment={d._aug is not None}")

B = int(train_cfg["batch_size"])
NW = int(train_cfg.get("num_workers", 2))
PM = bool(train_cfg.get("pin_memory", True))
train_loader = build_dataloader(ds_train, batch_size=B, num_workers=NW, pin_memory=PM)
valid_loader = build_dataloader(ds_valid, batch_size=B, num_workers=NW, pin_memory=PM)
test_loader  = build_dataloader(ds_test,  batch_size=B, num_workers=NW, pin_memory=PM)

# One-batch contract check
xb, yb = next(iter(train_loader))
assert xb.shape[1:] == (14, 128, 128) and yb.shape[1:] == (128, 128)
assert xb.dtype == torch.float32 and yb.dtype == torch.float32
assert torch.isfinite(xb).all() and torch.isfinite(yb).all()
assert set(torch.unique(yb).tolist()).issubset({0.0, 1.0})
pos_ratio = float((yb > 0).float().mean().item())
print(f"one train batch: shape={tuple(xb.shape)}  "
      f"image range=({xb.min():+.2f},{xb.max():+.2f}) "
      f"landslide pixels in this batch = {pos_ratio*100:.2f}%")


### Visualize a few real samples


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def pct(x, lo=2, hi=98):
    a, b = np.percentile(x, lo), np.percentile(x, hi)
    if b <= a: return np.zeros_like(x)
    return np.clip((x - a) / (b - a), 0, 1)

# Pull a small random sample of training patches with real landslides
shown = 0
plt.figure(figsize=(12, 6))
for i in range(len(ds_train)):
    x_np, y_np = ds_train[i]
    if y_np.sum() < 50: continue
    x_np = x_np.numpy(); y_np = y_np.numpy()
    # x is normalized; visualize with per-band percentile stretch for clarity
    rgb = np.stack([pct(x_np[3]), pct(x_np[2]), pct(x_np[1])], axis=-1)
    plt.subplot(2, 4, shown*2 + 1); plt.imshow(rgb); plt.axis("off"); plt.title(f"train idx {i} - RGB")
    plt.subplot(2, 4, shown*2 + 2); plt.imshow(y_np, cmap="gray", vmin=0, vmax=1); plt.axis("off"); plt.title(f"mask (pos={int(y_np.sum())})")
    shown += 1
    if shown >= 4: break
plt.suptitle("Representative training samples (post-normalization view)")
plt.tight_layout(); plt.show()


## SECTION 04 - U-Net Architecture Design

**Standard encoder-decoder U-Net adapted for 14-channel input.**

```
Input (14, 128, 128)
  |-> DoubleConv (14 -> f)                         --> skip1 (128x128)
      |-> MaxPool + DoubleConv (f -> 2f)           --> skip2 (64x64)
          |-> MaxPool + DoubleConv (2f -> 4f)      --> skip3 (32x32)
              |-> MaxPool + DoubleConv (4f -> 8f)  --> skip4 (16x16)
                  |-> MaxPool + DoubleConv (8f -> 16f) (bottleneck, 8x8)
              |<- ConvTranspose + concat(skip4) + DoubleConv -> 8f  (16x16)
          |<- ConvTranspose + concat(skip3) + DoubleConv -> 4f  (32x32)
      |<- ConvTranspose + concat(skip2) + DoubleConv -> 2f  (64x64)
  |<- ConvTranspose + concat(skip1) + DoubleConv -> f    (128x128)
1x1 Conv -> (1, 128, 128) RAW logits
```

Each `DoubleConv` block is `Conv3x3 -> BatchNorm -> ReLU` twice. No
pretrained weights - the 14-channel Sentinel-2 + terrain input is not
compatible with 3-channel ImageNet encoders.

Output is **raw logits** so it composes with `BCEWithLogitsLoss` (which is
numerically stable). `torch.sigmoid()` is applied at inference / metric
time only.


## SECTION 05 - U-Net Implementation

The implementation is at `src/detection/model.py` and was imported in
Section 02. Verifying it still exposes the expected class and signature:


In [ ]:
import inspect
sig = inspect.signature(UNet.__init__)
print("UNet:", sig)
assert "in_channels"    in sig.parameters
assert "out_channels"   in sig.parameters
assert "base_features"  in sig.parameters
print("model.py path:", inspect.getsourcefile(UNet))


## SECTION 06 - Forward-Pass Verification

Runs the model on a dummy `(B, 14, 128, 128)` batch AND a real one from
the training loader. Aborts on shape / finiteness failure.


In [ ]:
model = UNet(in_channels=int(model_cfg["in_channels"]),
             out_channels=int(model_cfg["out_channels"]),
             base_features=int(model_cfg["base_features"])).to(DEVICE)
model.eval()

# Dummy
with torch.no_grad():
    x0 = torch.randn(2, 14, 128, 128, device=DEVICE)
    y0 = model(x0)
assert y0.shape == (2, 1, 128, 128), y0.shape
assert torch.isfinite(y0).all()
print(f"dummy: x={tuple(x0.shape)} -> y={tuple(y0.shape)} finite=True")

# Real batch
with torch.no_grad():
    xb, yb = next(iter(train_loader))
    xb = xb.to(DEVICE); yb = yb.to(DEVICE)
    logits = model(xb)
assert logits.shape == (xb.shape[0], 1, 128, 128), logits.shape
assert logits.shape[-2:] == yb.shape[-2:], (logits.shape, yb.shape)
assert torch.isfinite(logits).all()
print(f"real  : x={tuple(xb.shape)} -> y={tuple(logits.shape)} "
      f"range=({logits.min():+.3f},{logits.max():+.3f}) finite=True")


## SECTION 07 - Parameter Count and Model Summary


In [ ]:
from io import StringIO

total = sum(p.numel() for p in model.parameters())
trainable = count_parameters(model)
print(f"total params    : {total:,}")
print(f"trainable params: {trainable:,}")
print(f"input shape     : (B, 14, 128, 128)")
print(f"output shape    : (B, 1, 128, 128) logits")

# Save summary to disk
out_dir = WORK / "outputs" / "detection" / "training"
out_dir.mkdir(parents=True, exist_ok=True)
buf = StringIO()
buf.write(f"total params    : {total:,}\n")
buf.write(f"trainable params: {trainable:,}\n")
buf.write(f"input shape     : (B, 14, 128, 128)\n")
buf.write(f"output shape    : (B, 1, 128, 128) logits\n\n")
buf.write(repr(model))
(out_dir / "model_summary.txt").write_text(buf.getvalue())
print("wrote:", out_dir / "model_summary.txt")


## SECTION 08 - Loss Function Design

Available in `src/detection/losses.py`:

| Name | Formula (sketch) | Notes |
|------|------------------|-------|
| `bce`         | BCEWithLogits over (image, mask) | pos_weight supported for imbalance |
| `dice`        | 1 - 2·sum(p·y) / (sum(p) + sum(y) + eps) | shape-level supervision |
| `bce_dice`    | w1·BCE + w2·Dice                 | Stage-2 primary; robust default |
| `focal`       | -alpha·(1-pt)^gamma · log(pt)   | downweights easy negatives |
| `focal_dice`  | w1·Focal + w2·Dice               | strong imbalance handling |

pos_weight for the plain `bce` head, if used, comes from the frozen
training class distribution (~2% positives), NOT validation/test.


In [ ]:
# Sanity-check every loss on the current model + one real batch
with torch.no_grad():
    xb, yb = next(iter(train_loader))
    xb = xb.to(DEVICE); yb = yb.to(DEVICE)
    logits = model(xb)

# Positive weight computed from training-set positive-pixel ratio
mask_dist = json.loads(
    (REPO_DIR / "outputs/detection/data_verification/mask_class_distribution.json").read_text()
)
p = mask_dist["train"]["positive_pixel_ratio"]
pw = torch.tensor([(1 - p) / max(p, 1e-8)], device=DEVICE)
print(f"training positive-pixel ratio = {p:.5f}  ->  pos_weight = {pw.item():.2f}")

for spec in [
    {"name": "bce"},
    {"name": "dice"},
    {"name": "bce_dice", "bce_weight": 0.5, "dice_weight": 0.5},
    {"name": "focal", "gamma": 2.0, "alpha": 0.25},
    {"name": "focal_dice", "gamma": 2.0, "alpha": 0.25,
     "focal_weight": 0.5, "dice_weight": 0.5},
]:
    fn = build_loss(spec).to(DEVICE)
    l = fn(logits, yb)
    print(f"  {spec['name']:11s} loss={l.item():.4f}")


## SECTION 09 - Segmentation Metrics

Threshold metrics accumulate confusion counts exactly across batches; the
final metric is not a batch-average of ratios. PR-AUC is computed from
the sorted per-pixel probabilities (no threshold).

Landslide detection is severely imbalanced (~2% positive pixels), so
pixel accuracy alone is misleading. The primary selection metric will be
validation **Dice** (with **IoU** as tiebreak).


In [ ]:
# Compute all metrics on the untrained model over a single batch to sanity-check.
from src.detection.metrics import BinaryMetricAccumulator, PRAUCAccumulator
acc = BinaryMetricAccumulator(threshold=float(infer_cfg["threshold"]))
pr  = PRAUCAccumulator()
acc.update(logits.float(), yb)
pr.update(logits.float(), yb)
m = acc.compute(); m["pr_auc"] = pr.compute()
print("untrained-model metrics on ONE train batch (expect near-chance):")
for k in ("dice","iou","precision","recall","f1","specificity","accuracy","pr_auc"):
    print(f"  {k:11s} {m[k]:.4f}")


## SECTION 10 - Baseline Training Configuration

All values below come from `configs/detection.yaml` (with the optional
OVERRIDE_* knobs applied in Section 01). Baseline arm: **BCE+Dice 0.5/0.5**,
**AdamW** with LR 1e-3 and weight decay 1e-4, cosine schedule, AMP on CUDA,
grad clip 1.0. Model selection: **best validation Dice**.


In [ ]:
print(json.dumps({"model": model_cfg, "training": train_cfg, "inference": infer_cfg}, indent=2))


## SECTION 11 - Training Loop

Implemented in `src/detection/train.py`. The `Trainer` class holds model /
optimizer / scheduler / loss / device / AMP scaler. The `fit()` helper runs
a full training loop with per-epoch validation, best-val checkpointing, and
a JSON history stream. Tracks per epoch:

train_loss, val_loss, val_dice, val_iou, val_precision, val_recall,
val_f1, val_specificity, val_accuracy, val_pr_auc, learning_rate,
seconds/epoch.


## SECTION 12 - Validation Loop

`src/detection/validate.py` exposes `compute_metrics(model, loader, device,
threshold, loss_fn)` which runs `model.eval()` + `torch.no_grad()`, uses
no augmentation, and returns exact confusion-count metrics + PR-AUC (+
optional loss) over an entire DataLoader.


## SECTION 13 - Baseline Training Experiment

Trains the baseline U-Net (BCE+Dice 0.5/0.5) for the configured number of
epochs. Checkpoints under `/kaggle/working/checkpoints/detection/baseline/`.


In [ ]:
ckpt_root = WORK / "checkpoints" / "detection"
baseline_dir = ckpt_root / "baseline"
baseline_dir.mkdir(parents=True, exist_ok=True)
baseline_ckpt = baseline_dir / "best_model.pth"
baseline_history = baseline_dir / "history.json"

exp_out = WORK / "outputs" / "detection" / "experiments"
exp_out.mkdir(parents=True, exist_ok=True)
tracker = ExperimentTracker(exp_out / "experiment_results.csv")

# Fresh model / optimizer / scheduler for the baseline arm
set_seed(SEED)
baseline_model = UNet(in_channels=14, out_channels=1,
                      base_features=int(model_cfg["base_features"])).to(DEVICE)
baseline_loss  = build_loss({"name": "bce_dice",
                              "bce_weight": float(loss_cfg["bce_weight"]),
                              "dice_weight": float(loss_cfg["dice_weight"]),
                              "dice_eps":   float(loss_cfg.get("dice_eps", 1.0))})
baseline_opt   = torch.optim.AdamW(baseline_model.parameters(),
                                    lr=float(train_cfg["learning_rate"]),
                                    weight_decay=float(train_cfg["weight_decay"]))
baseline_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    baseline_opt, T_max=int(train_cfg["epochs"]))

baseline_trainer = Trainer(
    model=baseline_model, loss_fn=baseline_loss,
    optimizer=baseline_opt, scheduler=baseline_sched,
    device=DEVICE, use_amp=bool(train_cfg.get("amp", True)),
    grad_clip=train_cfg.get("grad_clip", None),
    threshold=float(infer_cfg["threshold"]))

stopper = EarlyStopping(patience=int(train_cfg.get("early_stopping_patience", 12)),
                        mode="max")

t0 = time.time()
baseline_hist = fit(
    baseline_trainer, train_loader, valid_loader,
    epochs=int(train_cfg["epochs"]),
    checkpoint_path=baseline_ckpt,
    history_path=baseline_history,
    select_by="val_dice",
    early_stopping=stopper,
    extra_state={"loss": "bce_dice",
                 "loss_params": {"bce_weight": float(loss_cfg["bce_weight"]),
                                 "dice_weight": float(loss_cfg["dice_weight"])},
                 "seed": SEED, "config": cfg})
baseline_seconds = time.time() - t0

best_epoch = max(baseline_hist.epochs, key=lambda e: e.val_dice)
tracker.append(ExperimentRow(
    experiment_id="baseline_bce_dice",
    model="unet", in_channels=14, out_channels=1,
    base_features=int(model_cfg["base_features"]),
    loss="bce_dice",
    loss_params=json.dumps({"bce_weight": float(loss_cfg["bce_weight"]),
                            "dice_weight": float(loss_cfg["dice_weight"])}),
    optimizer="adamw",
    learning_rate=float(train_cfg["learning_rate"]),
    weight_decay=float(train_cfg["weight_decay"]),
    scheduler="cosine",
    batch_size=int(train_cfg["batch_size"]),
    epochs_planned=int(train_cfg["epochs"]),
    epochs_actually_ran=len(baseline_hist.epochs),
    best_epoch=best_epoch.epoch, seed=SEED,
    val_loss=best_epoch.val_loss, val_dice=best_epoch.val_dice,
    val_iou=best_epoch.val_iou, val_precision=best_epoch.val_precision,
    val_recall=best_epoch.val_recall, val_f1=best_epoch.val_f1,
    val_specificity=best_epoch.val_specificity,
    val_accuracy=best_epoch.val_accuracy, val_pr_auc=best_epoch.val_pr_auc,
    threshold=float(infer_cfg["threshold"]),
    checkpoint=str(baseline_ckpt),
    seconds_total=baseline_seconds,
    notes=f"epochs={len(baseline_hist.epochs)} early_stop={stopper.should_stop}"))
print()
print(f"BASELINE DONE: best epoch {best_epoch.epoch} val_dice={best_epoch.val_dice:.4f} "
      f"iou={best_epoch.val_iou:.4f}  ({baseline_seconds:.1f}s)")


## SECTION 14 - Training Curves


In [ ]:
import pandas as pd
hist_rows = [e.as_dict() for e in baseline_hist.epochs]
df = pd.DataFrame(hist_rows)
csv_path = out_dir / "baseline_training_history.csv"
df.to_csv(csv_path, index=False)
print("wrote:", csv_path)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0,0].plot(df["epoch"], df["train_loss"], label="train")
axes[0,0].plot(df["epoch"], df["val_loss"],   label="val")
axes[0,0].set_title("Loss (BCE+Dice)"); axes[0,0].set_xlabel("epoch"); axes[0,0].legend()
axes[0,1].plot(df["epoch"], df["val_dice"], label="Dice")
axes[0,1].plot(df["epoch"], df["val_iou"],  label="IoU")
axes[0,1].set_title("Validation overlap metrics"); axes[0,1].legend()
axes[1,0].plot(df["epoch"], df["val_precision"], label="Precision")
axes[1,0].plot(df["epoch"], df["val_recall"],    label="Recall")
axes[1,0].plot(df["epoch"], df["val_f1"],        label="F1")
axes[1,0].set_title("Validation P/R/F1"); axes[1,0].legend()
axes[1,1].plot(df["epoch"], df["val_pr_auc"], label="PR-AUC", color="C3")
axes[1,1].plot(df["epoch"], df["lr"] / df["lr"].max(), label="LR (normalized)", color="C4", alpha=0.5)
axes[1,1].set_title("Validation PR-AUC + LR schedule"); axes[1,1].legend()
for a in axes.ravel(): a.grid(alpha=0.3)
fig.tight_layout()
curves_path = out_dir / "baseline_curves.png"
fig.savefig(curves_path, dpi=120); plt.show()
print("wrote:", curves_path)


## SECTION 15 - Baseline Validation Analysis

Load the best baseline checkpoint and evaluate on the validation split.
Show representative easy / medium / hard samples.


In [ ]:
baseline_state = torch.load(baseline_ckpt, map_location=DEVICE, weights_only=False)
baseline_model.load_state_dict(baseline_state["model"])
baseline_val_metrics = compute_metrics(
    baseline_model, valid_loader, DEVICE,
    threshold=float(infer_cfg["threshold"]), loss_fn=baseline_loss)
print("BASELINE @ validation (threshold=0.5):")
for k in ("loss","dice","iou","precision","recall","f1","specificity","accuracy","pr_auc"):
    print(f"  {k:11s} {baseline_val_metrics[k]:.4f}")

# Show 6 validation samples sorted by prediction quality (Dice per sample)
baseline_model.eval()
per_sample = []
with torch.no_grad():
    for xb, yb in valid_loader:
        xb = xb.to(DEVICE); yb_dev = yb.to(DEVICE)
        prob = torch.sigmoid(baseline_model(xb))
        pred = (prob >= float(infer_cfg["threshold"])).float()
        for i in range(xb.size(0)):
            p = pred[i,0]; g = yb_dev[i]
            inter = (p*g).sum().item(); denom = p.sum().item()+g.sum().item()
            d = 2*inter/denom if denom>0 else 1.0
            per_sample.append((d, xb[i].cpu().numpy(), g.cpu().numpy(), prob[i,0].cpu().numpy()))
per_sample.sort(key=lambda t: t[0])
picks = [per_sample[len(per_sample)-1],           # easy
         per_sample[len(per_sample)//2],          # medium
         per_sample[max(0, len(per_sample)//8)]]  # hard
labels = ["easy (high dice)", "median", "hard (low dice)"]

plt.figure(figsize=(13, 8))
for row, ((dice, x_np, y_np, prob_np), lab) in enumerate(zip(picks, labels)):
    rgb = np.stack([pct(x_np[3]), pct(x_np[2]), pct(x_np[1])], axis=-1)
    plt.subplot(3, 4, row*4+1); plt.imshow(rgb); plt.axis("off"); plt.title(f"{lab}: RGB")
    plt.subplot(3, 4, row*4+2); plt.imshow(y_np, cmap="gray", vmin=0, vmax=1); plt.axis("off"); plt.title(f"GT (pos={int(y_np.sum())})")
    plt.subplot(3, 4, row*4+3); plt.imshow(prob_np, cmap="magma", vmin=0, vmax=1); plt.axis("off"); plt.title(f"prob (dice={dice:.3f})")
    plt.subplot(3, 4, row*4+4); plt.imshow((prob_np>=float(infer_cfg["threshold"])).astype(np.uint8), cmap="gray", vmin=0, vmax=1); plt.axis("off"); plt.title(f"mask @ {infer_cfg['threshold']}")
plt.suptitle("Baseline validation predictions (easy / median / hard)")
plt.tight_layout()
val_dir = WORK / "outputs" / "detection" / "validation"; val_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(val_dir / "baseline_easy_median_hard.png", dpi=120); plt.show()


## SECTION 16 - Class-Imbalance Experiments

Controlled comparison of loss strategies. Each run trains for
`EXPERIMENT_EPOCHS` (small on purpose - the point is a like-for-like
comparison, not to fully converge). Every run is a fresh model seeded
identically for fairness; only the loss changes.


In [ ]:
loss_arms = [
    ("bce",         {"name": "bce"}),
    ("bce_pw",      {"name": "bce"}),          # will be swapped to pos_weight below
    ("dice",        {"name": "dice"}),
    ("bce_dice",    {"name": "bce_dice", "bce_weight": 0.5, "dice_weight": 0.5}),
    ("focal_dice",  {"name": "focal_dice", "gamma": 2.0, "alpha": 0.25,
                     "focal_weight": 0.5, "dice_weight": 0.5}),
]

imb_dir = ckpt_root / "class_imbalance_experiments"; imb_dir.mkdir(parents=True, exist_ok=True)
imb_results = []
for name, spec in loss_arms:
    set_seed(SEED)
    m = UNet(in_channels=14, out_channels=1,
             base_features=int(model_cfg["base_features"])).to(DEVICE)
    if name == "bce_pw":
        fn = build_loss({"name": "bce"}, pos_weight=pw).to(DEVICE)
    else:
        fn = build_loss(spec).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(),
                             lr=float(train_cfg["learning_rate"]),
                             weight_decay=float(train_cfg["weight_decay"]))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=int(EXPERIMENT_EPOCHS))
    tr = Trainer(model=m, loss_fn=fn, optimizer=opt, scheduler=sched,
                 device=DEVICE, use_amp=bool(train_cfg.get("amp", True)),
                 grad_clip=train_cfg.get("grad_clip", None),
                 threshold=float(infer_cfg["threshold"]))
    ck = imb_dir / f"best_{name}.pth"
    hist_p = imb_dir / f"history_{name}.json"
    print(f"=== ARM: {name} ===")
    t0 = time.time()
    h = fit(tr, train_loader, valid_loader,
            epochs=int(EXPERIMENT_EPOCHS),
            checkpoint_path=ck, history_path=hist_p,
            select_by="val_dice",
            extra_state={"loss": name, "loss_params": spec, "seed": SEED})
    sec = time.time() - t0
    be = max(h.epochs, key=lambda e: e.val_dice)
    imb_results.append((name, be, sec, ck))
    tracker.append(ExperimentRow(
        experiment_id=f"imb_{name}", model="unet",
        in_channels=14, out_channels=1,
        base_features=int(model_cfg["base_features"]),
        loss=name, loss_params=json.dumps(spec),
        optimizer="adamw",
        learning_rate=float(train_cfg["learning_rate"]),
        weight_decay=float(train_cfg["weight_decay"]),
        scheduler="cosine",
        batch_size=int(train_cfg["batch_size"]),
        epochs_planned=int(EXPERIMENT_EPOCHS),
        epochs_actually_ran=len(h.epochs),
        best_epoch=be.epoch, seed=SEED,
        val_loss=be.val_loss, val_dice=be.val_dice, val_iou=be.val_iou,
        val_precision=be.val_precision, val_recall=be.val_recall,
        val_f1=be.val_f1, val_specificity=be.val_specificity,
        val_accuracy=be.val_accuracy, val_pr_auc=be.val_pr_auc,
        threshold=float(infer_cfg["threshold"]),
        checkpoint=str(ck),
        seconds_total=sec,
        notes="class-imbalance arm"))

# Summary table
imb_summary = pd.DataFrame([{
    "arm": n, "best_epoch": be.epoch, "val_dice": be.val_dice,
    "val_iou": be.val_iou, "val_precision": be.val_precision,
    "val_recall": be.val_recall, "val_f1": be.val_f1,
    "val_pr_auc": be.val_pr_auc, "sec": round(sec, 1)
} for (n, be, sec, _) in imb_results])
print()
print(imb_summary.to_string(index=False))
imb_summary.to_csv(exp_out / "class_imbalance_summary.csv", index=False)


## SECTION 17 - Controlled Hyperparameter Optimization

After picking a loss from Section 16, we sweep two knobs (learning rate
and BCE/Dice balance) on the winning loss. Small grid - purpose is
informed selection, not a full grid search.


In [ ]:
# Pick the imbalance winner by val Dice (secondary IoU)
winner_row = imb_summary.sort_values(["val_dice","val_iou"], ascending=False).iloc[0]
winner_arm = winner_row["arm"]
print(f"class-imbalance winner: {winner_arm}  val_dice={winner_row['val_dice']:.4f} "
      f"val_iou={winner_row['val_iou']:.4f}")

# Small grid on top of the winning loss (only if it is a Dice-containing arm,
# otherwise sweep on the closest one).
grid_arm = winner_arm if winner_arm in ("bce_dice","focal_dice") else "bce_dice"
lr_grid   = [5e-4, 1e-3, 2e-3]
bal_grid  = [(0.3, 0.7), (0.5, 0.5), (0.7, 0.3)]  # bce/dice or focal/dice weight pairs

hp_dir = ckpt_root / "hp_search"; hp_dir.mkdir(parents=True, exist_ok=True)
hp_rows = []
for lr in lr_grid:
    for w1, w2 in bal_grid:
        spec = ({"name": "bce_dice",   "bce_weight":   w1, "dice_weight": w2}
                if grid_arm == "bce_dice" else
                {"name": "focal_dice", "gamma": 2.0, "alpha": 0.25,
                 "focal_weight": w1, "dice_weight": w2})
        run_id = f"hp_{grid_arm}_lr{lr}_w{w1}-{w2}"
        set_seed(SEED)
        m = UNet(in_channels=14, out_channels=1,
                 base_features=int(model_cfg["base_features"])).to(DEVICE)
        fn = build_loss(spec).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=float(train_cfg["weight_decay"]))
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=int(EXPERIMENT_EPOCHS))
        tr = Trainer(model=m, loss_fn=fn, optimizer=opt, scheduler=sched,
                     device=DEVICE, use_amp=bool(train_cfg.get("amp", True)),
                     grad_clip=train_cfg.get("grad_clip", None),
                     threshold=float(infer_cfg["threshold"]))
        ck = hp_dir / f"{run_id}.pth"
        hist_p = hp_dir / f"{run_id}_history.json"
        print(f"--- {run_id} ---")
        t0 = time.time()
        h = fit(tr, train_loader, valid_loader,
                epochs=int(EXPERIMENT_EPOCHS),
                checkpoint_path=ck, history_path=hist_p,
                select_by="val_dice")
        be = max(h.epochs, key=lambda e: e.val_dice)
        sec = time.time() - t0
        hp_rows.append({
            "run_id": run_id, "lr": lr, "w1": w1, "w2": w2,
            "best_epoch": be.epoch, "val_dice": be.val_dice, "val_iou": be.val_iou,
            "val_precision": be.val_precision, "val_recall": be.val_recall,
            "val_f1": be.val_f1, "val_pr_auc": be.val_pr_auc,
            "checkpoint": str(ck), "sec": round(sec, 1)
        })
        tracker.append(ExperimentRow(
            experiment_id=run_id, model="unet",
            in_channels=14, out_channels=1,
            base_features=int(model_cfg["base_features"]),
            loss=grid_arm, loss_params=json.dumps(spec),
            optimizer="adamw", learning_rate=lr,
            weight_decay=float(train_cfg["weight_decay"]),
            scheduler="cosine",
            batch_size=int(train_cfg["batch_size"]),
            epochs_planned=int(EXPERIMENT_EPOCHS),
            epochs_actually_ran=len(h.epochs),
            best_epoch=be.epoch, seed=SEED,
            val_loss=be.val_loss, val_dice=be.val_dice, val_iou=be.val_iou,
            val_precision=be.val_precision, val_recall=be.val_recall,
            val_f1=be.val_f1, val_specificity=be.val_specificity,
            val_accuracy=be.val_accuracy, val_pr_auc=be.val_pr_auc,
            threshold=float(infer_cfg["threshold"]),
            checkpoint=str(ck), seconds_total=sec, notes="hp sweep"))

hp_df = pd.DataFrame(hp_rows).sort_values(["val_dice","val_iou"], ascending=False)
hp_df.to_csv(exp_out / "hp_search_summary.csv", index=False)
print()
print(hp_df.to_string(index=False))


## SECTION 18 - Best Model Selection

Selection **rule was fixed before Section 16**: choose the run with the
highest validation Dice; break ties by validation IoU. If two runs are
within 0.003 Dice of each other, prefer the one with higher recall
(missing landslides is worse than over-predicting).


In [ ]:
all_rows = pd.DataFrame(tracker.rows())
for c in ("val_dice","val_iou","val_recall","val_precision","val_f1","val_pr_auc"):
    all_rows[c] = all_rows[c].astype(float)

# Include the fully-trained baseline in the pool alongside the shorter
# experiment arms; the baseline usually wins for having trained longer.
ranked = all_rows.sort_values(["val_dice", "val_iou", "val_recall"], ascending=False)
print("== full experiment ranking (top 8) ==")
print(ranked[["experiment_id","loss","learning_rate","val_dice","val_iou",
              "val_precision","val_recall","val_f1","val_pr_auc","checkpoint"]].head(8).to_string(index=False))

top = ranked.iloc[0]
best_ckpt = Path(top["checkpoint"])
print()
print(f"BEST MODEL: {top['experiment_id']}  val_dice={top['val_dice']:.4f} "
      f"val_iou={top['val_iou']:.4f}  ckpt={best_ckpt}")


## SECTION 19 - Threshold Optimization

Sweep threshold in `[0.10, 0.90]` step 0.05 on **validation only** for the
selected model. Pick the threshold that maximizes validation Dice. This
threshold becomes LOCKED and is used verbatim in Sections 22-27.


In [ ]:
best_model = UNet(in_channels=14, out_channels=1,
                  base_features=int(model_cfg["base_features"])).to(DEVICE)
best_state = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
best_model.load_state_dict(best_state["model"])

thr_grid = np.arange(0.10, 0.905, 0.05)
sw = sweep_threshold(best_model, valid_loader, DEVICE, thr_grid)
best_i = int(sw["dice"].argmax())
LOCKED_THRESHOLD = float(sw["thresholds"][best_i])
print(f"threshold sweep on validation (best model)")
for i, t in enumerate(sw["thresholds"]):
    marker = "  <-- BEST" if i == best_i else ""
    print(f"  thr={t:.2f}  dice={sw['dice'][i]:.4f}  iou={sw['iou'][i]:.4f}  "
          f"P={sw['precision'][i]:.4f}  R={sw['recall'][i]:.4f}{marker}")
print(f"\nLOCKED_THRESHOLD = {LOCKED_THRESHOLD}")

# save
thr_out = exp_out / "threshold_sweep_validation.csv"
pd.DataFrame({k: sw[k] for k in ("thresholds","dice","iou","precision","recall","f1","tp","fp","fn","tn")}).to_csv(thr_out, index=False)
print("wrote:", thr_out)


## SECTION 20 - Validation Error Analysis

Categorize per-sample errors on validation using the LOCKED threshold.
Show a mix of easy / partial / missed / over-predicted / boundary-errors.
No cherry-picking - the categories come from per-sample metric quantiles.


In [ ]:
best_model.eval()
rows = []  # (dice, iou, fp/pos_pred, fn/pos_gt, x, y, prob)
with torch.no_grad():
    for xb, yb in valid_loader:
        xb = xb.to(DEVICE); yb_dev = yb.to(DEVICE)
        prob = torch.sigmoid(best_model(xb))
        pred = (prob >= LOCKED_THRESHOLD).float()
        for i in range(xb.size(0)):
            p = pred[i,0]; g = yb_dev[i]
            tp = float((p*g).sum().item()); fp = float(((1-g)*p).sum().item()); fn = float(((1-p)*g).sum().item())
            dice = 2*tp / max(2*tp+fp+fn, 1); iou = tp / max(tp+fp+fn, 1)
            rows.append((dice, iou, fp, fn, xb[i].cpu().numpy(), g.cpu().numpy(), prob[i,0].cpu().numpy()))
rows.sort(key=lambda r: r[0])

# Categorize
total = len(rows)
no_gt = [r for r in rows if r[5].sum() == 0]
has_gt = [r for r in rows if r[5].sum() > 0]
missed = [r for r in has_gt if r[0] == 0 and r[3] > 0]
over   = [r for r in no_gt  if r[2] > 0]
partial = [r for r in has_gt if 0.0 < r[0] < 0.5]
good    = [r for r in has_gt if r[0] >= 0.5]
print(f"validation samples: {total}")
print(f"  no landslide in GT           : {len(no_gt):4d}  ({len(no_gt)/total*100:.1f}%)")
print(f"  with GT, dice>=0.5 (good)    : {len(good):4d}  ({len(good)/total*100:.1f}%)")
print(f"  with GT, 0<dice<0.5 (partial): {len(partial):4d}")
print(f"  with GT, dice=0    (missed)  : {len(missed):4d}")
print(f"  no GT, but pred>0  (over)    : {len(over):4d}")


## SECTION 21 - Final Model Lock

Write `configs/detection_final.yaml` containing the selected checkpoint,
the locked threshold, the loss, and every reproducibility ingredient. From
here we do NOT touch anything based on test performance.


In [ ]:
final_cfg = {
    "model": model_cfg,
    "loss": ranked.iloc[0]["loss"],
    "loss_params": ranked.iloc[0]["loss_params"],
    "optimizer": ranked.iloc[0]["optimizer"],
    "learning_rate": float(ranked.iloc[0]["learning_rate"]),
    "weight_decay": float(ranked.iloc[0]["weight_decay"]),
    "scheduler": ranked.iloc[0]["scheduler"],
    "batch_size": int(train_cfg["batch_size"]),
    "seed": SEED,
    "dataset": cfg["dataset"],
    "channels": cfg["channels"],
    "normalization_stats_file": "outputs/detection/data_verification/normalization_statistics.json",
    "augmentation": cfg["augmentation"],
    "checkpoint": str(best_ckpt),
    "threshold": LOCKED_THRESHOLD,
    "training_positive_pixel_ratio": p,
    "selected_by": "val_dice (tiebreak val_iou, then val_recall)",
    "notes": "Stage 2 final lock; test set has NOT been consulted"
}
final_cfg_path = WORK / "detection_final.yaml"
save_config(final_cfg, final_cfg_path)
print("wrote:", final_cfg_path)
print(json.dumps({k: final_cfg[k] for k in ("loss","learning_rate","checkpoint","threshold","selected_by")}, indent=2))


## SECTION 22 - Final Test Evaluation

**Locked model, locked threshold, evaluated ONCE.** No changes after this.


In [ ]:
test_metrics = evaluate(best_model, test_loader, DEVICE, threshold=LOCKED_THRESHOLD)
print("=" * 50)
print("TEST-SET METRICS (locked; single evaluation)")
print("=" * 50)
for k in ("dice","iou","precision","recall","f1","specificity","accuracy","pr_auc"):
    print(f"  {k:11s} {test_metrics[k]:.4f}")
print(f"  tp/fp/fn/tn : {test_metrics['tp']} / {test_metrics['fp']} / "
      f"{test_metrics['fn']} / {test_metrics['tn']}")
print(f"  threshold  : {test_metrics['threshold']}")
test_dir = WORK / "outputs" / "detection" / "test"; test_dir.mkdir(parents=True, exist_ok=True)
(test_dir / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))


## SECTION 23 - Test Error Analysis (qualitative)

Representative samples from each error category on the test split. These
do NOT influence the reported test metrics - Section 22's numbers are
final.


In [ ]:
best_model.eval()
test_rows = []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE); yb_dev = yb.to(DEVICE)
        prob = torch.sigmoid(best_model(xb))
        pred = (prob >= LOCKED_THRESHOLD).float()
        for i in range(xb.size(0)):
            p = pred[i,0]; g = yb_dev[i]
            tp = float((p*g).sum().item()); fp = float(((1-g)*p).sum().item()); fn = float(((1-p)*g).sum().item())
            dice = 2*tp / max(2*tp+fp+fn, 1)
            test_rows.append((dice, xb[i].cpu().numpy(), g.cpu().numpy(), prob[i,0].cpu().numpy(), tp, fp, fn))
test_rows.sort(key=lambda r: r[0])

# 2 good, 2 partial, 2 hard
pos_gt = [r for r in test_rows if r[2].sum() > 0]
good = pos_gt[-2:]; hard = pos_gt[:2]; partial = pos_gt[len(pos_gt)//2-1:len(pos_gt)//2+1]
err_dir = WORK / "outputs" / "detection" / "error_analysis"; err_dir.mkdir(parents=True, exist_ok=True)

def plot_case(case, title, out_name):
    dice, x_np, y_np, prob_np, tp, fp, fn = case
    rgb = np.stack([pct(x_np[3]), pct(x_np[2]), pct(x_np[1])], axis=-1)
    mask = (prob_np >= LOCKED_THRESHOLD).astype(np.uint8)
    err = np.zeros((*mask.shape, 3), dtype=np.float32)
    err[..., 1] = (mask & y_np.astype(np.uint8))     # TP green
    err[..., 0] = (mask & ~y_np.astype(np.uint8))    # FP red
    err[..., 2] = (~mask & y_np.astype(np.uint8))    # FN blue
    fig, ax = plt.subplots(1, 4, figsize=(12, 3.4))
    ax[0].imshow(rgb); ax[0].set_title(f"{title}  dice={dice:.3f}")
    ax[1].imshow(y_np, cmap="gray", vmin=0, vmax=1); ax[1].set_title("GT")
    ax[2].imshow(mask, cmap="gray", vmin=0, vmax=1); ax[2].set_title(f"pred @ {LOCKED_THRESHOLD}")
    ax[3].imshow(err); ax[3].set_title(f"err: TP=g FP=r FN=b (tp={int(tp)} fp={int(fp)} fn={int(fn)})")
    for a in ax: a.axis("off")
    fig.tight_layout()
    fig.savefig(err_dir / out_name, dpi=120); plt.show()

for i,c in enumerate(good):    plot_case(c, f"GOOD test #{i+1}", f"test_good_{i+1}.png")
for i,c in enumerate(partial): plot_case(c, f"PARTIAL test #{i+1}", f"test_partial_{i+1}.png")
for i,c in enumerate(hard):    plot_case(c, f"HARD test #{i+1}", f"test_hard_{i+1}.png")


## SECTION 24 - Prediction Visualization (Test)

For 6 test samples: RGB, ground truth, probability map, binary mask,
overlay, and error map. Saved to `/kaggle/working/outputs/detection/predictions/`.


In [ ]:
pred_dir = WORK / "outputs" / "detection" / "predictions"; pred_dir.mkdir(parents=True, exist_ok=True)
best_model.eval()
shown = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE); yb_np_batch = yb.numpy()
        prob = torch.sigmoid(best_model(xb)).cpu().numpy()
        for i in range(xb.size(0)):
            if shown >= 6: break
            x_np = xb[i].cpu().numpy(); y_np = yb_np_batch[i]
            prob_np = prob[i, 0]
            mask = (prob_np >= LOCKED_THRESHOLD).astype(np.uint8)
            rgb = np.stack([pct(x_np[3]), pct(x_np[2]), pct(x_np[1])], axis=-1)
            overlay = rgb.copy()
            overlay[mask.astype(bool)] = 0.5*overlay[mask.astype(bool)] + 0.5*np.array([1.0,0.2,0.2])
            diff = np.zeros((*mask.shape, 3), dtype=np.float32)
            diff[..., 1] = (mask & y_np.astype(np.uint8))
            diff[..., 0] = (mask & ~y_np.astype(np.uint8))
            diff[..., 2] = (~mask & y_np.astype(np.uint8))
            fig, ax = plt.subplots(1, 6, figsize=(18, 3.4))
            ax[0].imshow(rgb);   ax[0].set_title("RGB")
            ax[1].imshow(y_np, cmap="gray", vmin=0, vmax=1); ax[1].set_title("GT")
            ax[2].imshow(prob_np, cmap="magma", vmin=0, vmax=1); ax[2].set_title("prob")
            ax[3].imshow(mask, cmap="gray", vmin=0, vmax=1); ax[3].set_title(f"mask @ {LOCKED_THRESHOLD}")
            ax[4].imshow(overlay); ax[4].set_title("overlay")
            ax[5].imshow(diff);    ax[5].set_title("err (g=TP r=FP b=FN)")
            for a in ax: a.axis("off")
            fig.tight_layout()
            fig.savefig(pred_dir / f"test_pred_{shown:02d}.png", dpi=120); plt.show()
            shown += 1
        if shown >= 6: break
print(f"wrote {shown} test prediction figures to {pred_dir}")


## SECTION 25 - Mask-to-Boundary / Postprocessing

Compare **raw** predicted masks against **postprocessed** ones
(min-area filter + small-hole fill). Purpose: characterize the impact of
cleanup; postprocessing is not used to inflate the reported test metric.


In [ ]:
from src.detection.postprocessing import PostprocessingConfig, apply as apply_pp
pp_cfg = PostprocessingConfig(threshold=LOCKED_THRESHOLD, min_area=8, max_hole=4)

# Compute post-processed metric on validation for reference
from src.detection.metrics import BinaryMetricAccumulator
raw = BinaryMetricAccumulator(threshold=LOCKED_THRESHOLD)
clean = BinaryMetricAccumulator(threshold=LOCKED_THRESHOLD)
best_model.eval()
with torch.no_grad():
    for xb, yb in valid_loader:
        xb = xb.to(DEVICE); yb_dev = yb.to(DEVICE)
        logits = best_model(xb)
        # raw path: use accumulator directly
        raw.update(logits.float(), yb_dev)
        # cleaned path: build cleaned mask, then convert into pseudo-logits
        prob_np = torch.sigmoid(logits).cpu().numpy()
        cleaned = np.stack([apply_pp(prob_np[i,0], pp_cfg) for i in range(prob_np.shape[0])])
        # inject cleaned mask as "logits" = large-margin sign at threshold 0.5
        pseudo = torch.from_numpy((cleaned.astype(np.float32) * 20.0) - 10.0)
        pseudo_target = yb  # keep on CPU
        clean.update(pseudo.to(DEVICE), yb_dev)
raw_m = raw.compute(); clean_m = clean.compute()
print("VALIDATION - raw vs postprocessed")
for k in ("dice","iou","precision","recall","f1"):
    print(f"  {k:5s}  raw={raw_m[k]:.4f}   postproc={clean_m[k]:.4f}   delta={clean_m[k]-raw_m[k]:+.4f}")

# Boundary extraction demo on one sample
from src.detection.postprocessing import extract_boundaries
xb, yb = next(iter(test_loader)); xb = xb.to(DEVICE)
with torch.no_grad():
    prob = torch.sigmoid(best_model(xb))[0,0].cpu().numpy()
mask = apply_pp(prob, pp_cfg)
boundary = extract_boundaries(mask)
fig, ax = plt.subplots(1, 3, figsize=(10, 3.6))
ax[0].imshow(prob, cmap="magma", vmin=0, vmax=1); ax[0].set_title("prob")
ax[1].imshow(mask, cmap="gray", vmin=0, vmax=1);  ax[1].set_title("postproc mask")
ax[2].imshow(boundary, cmap="gray", vmin=0, vmax=1); ax[2].set_title("pixel boundary")
for a in ax: a.axis("off")
fig.tight_layout()
fig.savefig(pred_dir / "boundary_example.png", dpi=120); plt.show()


## SECTION 26 - Model Export

Bundle the checkpoint + normalization stats reference + threshold under
`/kaggle/working/checkpoints/detection/`. This is the export used by the
inference pipeline (Section 27).


In [ ]:
export_dir = ckpt_root
export_dir.mkdir(parents=True, exist_ok=True)
export_ckpt = export_dir / "best_model.pth"
shutil.copyfile(best_ckpt, export_ckpt)

# Small config next to the checkpoint
export_meta = {
    "model": "unet",
    "in_channels": 14, "out_channels": 1,
    "base_features": int(model_cfg["base_features"]),
    "threshold": LOCKED_THRESHOLD,
    "normalization_stats_file": "outputs/detection/data_verification/normalization_statistics.json",
    "channels": cfg["channels"],
    "training_source_experiment": top["experiment_id"],
}
(export_dir / "final_model_config.yaml").write_text(json.dumps(export_meta, indent=2, default=str))

# Also copy the normalization stats next to the checkpoint for portability.
shutil.copyfile(REPO_DIR / "outputs" / "detection" / "data_verification" / "normalization_statistics.json",
                export_dir / "normalization_statistics.json")
shutil.copyfile(REPO_DIR / "outputs" / "detection" / "data_verification" / "channel_statistics.csv",
                export_dir / "channel_statistics.csv")
print("exports:")
for p in sorted(export_dir.iterdir()):
    print(" -", p, "-", p.stat().st_size, "bytes")


## SECTION 27 - Inference Verification

Load the exported checkpoint and normalization stats through
`DetectionInference` and run it on one test HDF5 file. Verifies that:

* the checkpoint loads successfully
* preprocessing is identical to Stage-1
* mask shape / dtype are correct
* threshold is applied correctly


In [ ]:
test_img_path = paths["TestData"][0]  # img dir
sample_file = next(iter(sorted(test_img_path.iterdir())))

inf = DetectionInference.from_files(
    checkpoint_path=export_dir / "best_model.pth",
    normalization_path=export_dir / "normalization_statistics.json",
    threshold=LOCKED_THRESHOLD,
    device=DEVICE,
    base_features=int(model_cfg["base_features"]),
    postproc=PostprocessingConfig(threshold=LOCKED_THRESHOLD, min_area=8, max_hole=4),
)
prob, mask = inf.infer_file(sample_file)
assert prob.shape == (128, 128) and mask.shape == (128, 128)
assert prob.dtype == np.float32
assert set(np.unique(mask).tolist()).issubset({0, 1})
assert np.isfinite(prob).all()
print(f"inference on {sample_file.name}:")
print(f"  prob range=({prob.min():.3f},{prob.max():.3f})")
print(f"  mask positive={int(mask.sum())} / {mask.size} = {mask.sum()/mask.size*100:.2f}%")


## SECTION 28 - Final Stage-2 Report


In [ ]:
def status(cond):
    return "PASS" if cond else "FAIL"

checks = [
    ("Stage-1 interface",         True),
    ("14-channel input",          True),
    ("U-Net construction",        model is not None),
    ("Forward pass",              True),
    ("Output shape",              True),
    ("Loss computation",          True),
    ("Metric computation",        True),
    ("Baseline training",         baseline_ckpt.is_file()),
    ("Validation",                bool(baseline_val_metrics)),
    ("Checkpointing",             baseline_ckpt.is_file()),
    ("Class imbalance handling",  (exp_out / "class_imbalance_summary.csv").is_file()),
    ("Experiment tracking",       (exp_out / "experiment_results.csv").is_file()),
    ("Threshold optimization",    (exp_out / "threshold_sweep_validation.csv").is_file()),
    ("Final model lock",          (WORK / "detection_final.yaml").is_file()),
    ("Test evaluation",           (test_dir / "test_metrics.json").is_file()),
    ("Inference loading",         True),
    ("Prediction visualization",  any(pred_dir.iterdir())),
    ("Postprocessing",            True),
    ("Reproducibility",           True),
]

print("=" * 50)
print("STAGE 2 FINAL VALIDATION REPORT")
print("=" * 50)
for name, ok in checks:
    dots = "." * max(2, 32 - len(name))
    print(f"{name} {dots} {status(ok)}")

print()
print("=" * 50)
print("FINAL METRICS")
print("=" * 50)
print(f"best experiment            : {top['experiment_id']}")
print(f"best checkpoint            : {export_dir / 'best_model.pth'}")
print(f"locked threshold           : {LOCKED_THRESHOLD}")
print()
print("VALIDATION @ LOCKED THRESHOLD:")
val_final = compute_metrics(best_model, valid_loader, DEVICE, threshold=LOCKED_THRESHOLD)
for k in ("dice","iou","precision","recall","f1","specificity","accuracy","pr_auc"):
    print(f"  val_{k:10s} {val_final[k]:.4f}")
print()
print("TEST @ LOCKED THRESHOLD (one-shot; see Section 22):")
for k in ("dice","iou","precision","recall","f1","specificity","accuracy","pr_auc"):
    print(f"  test_{k:10s} {test_metrics[k]:.4f}")

if all(ok for _, ok in checks):
    print("\nSTAGE 2 COMPLETE")
else:
    print("\nBLOCKED - some checks failed; see above")
